# Relax structure (MLP)

Relax atomic positions locally with ASE using **MACE-MP** today. Quantum ESPRESSO jobs are **not** used.

Authenticate once to **load** a Mat3ra material by name (or uploads + Standata) and **save** the relaxed structure back.

<h2 style="color:green">Usage</h2>

1. Set paths, material name, relax force tolerance, MACE presets, and tagging in sections **1.1–1.3**.
1. Authenticate (`OIDC_ACCESS_TOKEN`) and bind `ACCOUNT_ID`.
1. Local JSON lives under `../uploads`; platform mode searches by `MATERIAL_NAME`.
1. Click "Run" > "Run All".

## Summary

Install packages (+ PyTorch quirks in JupyterLite), pull material locally or API, visualize, optimize with ASE + MACE (+ `notebooks_utils` Plotly progress helpers), analyse optional heterostructure metrics, upload relaxed geometry.

**Switching calculators later** (different MLP wheels, constructor args): change the installer string in §1.1 and the calculator block in §4.1—the rest of the workflow stays ASE-shaped.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages
from mat3ra.notebooks_utils.primitive.environment import is_pyodide_environment

await install_packages("made|api_examples|torch|mace")

if is_pyodide_environment():
    from mat3ra.notebooks_utils.pyodide.packages.torch import apply_all_patches

    apply_all_patches()

### 1.2. Set parameters

In [ ]:
ORGANIZATION_NAME = None

FOLDER = "../uploads"
MATERIAL_NAME = "Silicon"  # used for local/Standata lookup and as platform search term

SUBSTRATE_TAG = 0
FILM_TAG = 1

### 1.3. Relaxation and MACE options

In [ ]:
# Final maximum force on any atom (eV/Å)
FMAX = 0.05

MACE_MODEL = "large"  # choose between "small", "medium", and "large"
MACE_DISPERSION = True  # enable D3 dispersion correction for van der Waals interactions
MACE_DEFAULT_DTYPE = "float32"  # floating-point precision for model inference; "float64" is more accurate but slower
MACE_DEVICE = "cpu"  # hardware target: "cpu" or "cuda" (GPU)

## 2. Authenticate and initialize API client
### 2.1. Authenticate

Browser login stores credentials in `OIDC_ACCESS_TOKEN`.

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"\u2705 Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

## 3. Load material
### 3.1. Resolve structure (platform API or uploads + Standata)

In [ ]:
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.ipython.entity.material.visualize import ViewersEnum, visualize_materials as visualize

material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME))

platform_matches = client.materials.list({"name": MATERIAL_NAME, "owner._id": ACCOUNT_ID})
if not platform_matches:
    raise ValueError(f"No material named {MATERIAL_NAME!r} found on platform for account {ACCOUNT_ID}")
material = Material.create(platform_matches[0])

visualize([{"material": material, "title": material.name}], viewer=ViewersEnum.wave)

### 3.2. Persist locally sourced structure
Runs deduplicated uploads via structural hash comparison when loading from uploads/Standata. Skipped automatically for `MATERIAL_INPUT_MODE == "platform"`.

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

material.basis.set_labels_from_list([])
input_saved = get_or_create_material(client, material, ACCOUNT_ID)
input_saved_material = Material.create(input_saved)
print("\u2705 Canonical input material record:", input_saved_material.id)


## 4. Apply relaxation
### 4.1. Relax with MACE (ASE)

In [ ]:
from mat3ra.notebooks_utils.ipython.plot._plotly import create_realtime_plot, create_update_callback
from mat3ra.notebooks_utils.primitive.environment import is_pyodide_environment
from mat3ra.notebooks_utils.pyodide.packages.mace import get_mace_model_pyodide
from mat3ra.made.tools.convert import to_ase
from mace.calculators import mace_mp
from ase.optimize import BFGS

mace_mp = mace_mp if not is_pyodide_environment() else get_mace_model_pyodide
calculator = mace_mp(
    model=MACE_MODEL,
    dispersion=MACE_DISPERSION,
    default_dtype=MACE_DEFAULT_DTYPE,
    device=MACE_DEVICE)

atoms_relaxed = to_ase(material)
atoms_relaxed.set_calculator(calculator)
dyn = BFGS(atoms_relaxed)

steps = []
energies = []
_progress_figure = create_realtime_plot(
    title="Real-time Optimization Progress",
    x_label="Step",
    y_label="Energy (eV)")
_progress_callback = create_update_callback(
    dynamic_object=dyn,
    value_getter=atoms_relaxed.get_total_energy,
    figure=_progress_figure,
    steps=steps,
    values=energies,
    print_format="Step: {}, Energy: {:.4f}",
)
dyn.attach(_progress_callback, interval=1)
dyn.run(fmax=FMAX)

ase_original_structure = to_ase(material)
ase_original_structure.set_calculator(calculator)
ase_relaxed_structure = atoms_relaxed

original_energy = ase_original_structure.get_total_energy()
relaxed_energy = atoms_relaxed.get_total_energy()

print(f"The final energy is {float(relaxed_energy):.3f} eV.")

## 5. Analyze results
### 5.1. View structure before and after relaxation

In [ ]:
from mat3ra.made.tools.convert import from_ase

material_original = Material.create(from_ase(ase_original_structure))
material_relaxed = Material.create(from_ase(ase_relaxed_structure))
material_original.name = material.name
material_relaxed.name = material.name + " (MACE relaxed)"

visualize(
    [
        {"material": material_original, "title": material_original.name},
        {"material": material_relaxed, "title": material_relaxed.name},
    ],
    viewer=ViewersEnum.wave,
)

### 5.2. Interlayer distance before and after relaxation
Requires substrate and film labels (atom tags), as produced when building interfaces with `mat3ra-made`.

In [ ]:
from mat3ra.made.tools.analyze.other import get_average_interlayer_distance

print(
    f"Interlayer distance before relaxation: {get_average_interlayer_distance(material_original, SUBSTRATE_TAG, FILM_TAG):.4f} Å")
print(
    f"Interlayer distance after relaxation:  {get_average_interlayer_distance(material_relaxed, SUBSTRATE_TAG, FILM_TAG):.4f} Å")

### 5.3. Interface energy decomposition
The structure should distinguish substrate vs film atoms by tags (e.g. 0 and 1) so components can be separated.

In [ ]:
def filter_atoms_by_tag(atoms, material_index):
    return atoms[atoms.get_tags() == material_index]


def calculate_energy(atoms, calc):
    atoms.set_calculator(calc)
    return atoms.get_total_energy()


def calculate_delta_energy(total_energy, *component_energies):
    return total_energy - sum(component_energies)


substrate_original = filter_atoms_by_tag(ase_original_structure, SUBSTRATE_TAG)
layer_original = filter_atoms_by_tag(ase_original_structure, FILM_TAG)
substrate_relaxed = filter_atoms_by_tag(ase_relaxed_structure, SUBSTRATE_TAG)
layer_relaxed = filter_atoms_by_tag(ase_relaxed_structure, FILM_TAG)

original_substrate_energy = calculate_energy(substrate_original, calculator)
original_layer_energy = calculate_energy(layer_original, calculator)
relaxed_substrate_energy = calculate_energy(substrate_relaxed, calculator)
relaxed_layer_energy = calculate_energy(layer_relaxed, calculator)

delta_original = calculate_delta_energy(original_energy, original_substrate_energy, original_layer_energy)
delta_relaxed = calculate_delta_energy(relaxed_energy, relaxed_substrate_energy, relaxed_layer_energy)

area = ase_original_structure.get_volume() / ase_original_structure.cell[2, 2]
n_interface = ase_relaxed_structure.get_global_number_of_atoms()
n_substrate = substrate_relaxed.get_global_number_of_atoms()
n_layer = layer_relaxed.get_global_number_of_atoms()
effective_delta_relaxed = (
                                  relaxed_energy / n_interface
                                  - (relaxed_substrate_energy / n_substrate + relaxed_layer_energy / n_layer)
                          ) / (2 * area)

print(f"Original Substrate energy: {original_substrate_energy:.4f} eV")
print(f"Relaxed Substrate energy:  {relaxed_substrate_energy:.4f} eV")
print(f"Original Layer energy:     {original_layer_energy:.4f} eV")
print(f"Relaxed Layer energy:      {relaxed_layer_energy:.4f} eV")
print("\nDelta between interface energy and sum of component energies")
print(f"Original Delta:            {delta_original:.4f} eV")
print(f"Relaxed Delta:             {delta_relaxed:.4f} eV")
print(f"Original Delta per area:   {delta_original / area:.4f} eV/Ang^2")
print(f"Relaxed Delta per area:    {delta_relaxed / area:.4f} eV/Ang^2")
print(f"Relaxed interface energy:  {relaxed_energy:.4f} eV")
print(
    f"Effective relaxed Delta per area: {effective_delta_relaxed:.4f} eV/Ang^2 ({effective_delta_relaxed / 0.16:.4f} J/m^2)")

## 6. Persist relaxed structure
### 6.1. Save relaxed geometry to Mat3ra (deduplicated)
Uses structural hashing (`get_or_create_material`) identical to QE workflow uploads.

In [ ]:
material_relaxed.basis.set_labels_from_list([])
_relaxed_saved_response = get_or_create_material(client, material_relaxed, ACCOUNT_ID)
relaxed_material_saved = Material.create(_relaxed_saved_response)
print(f"\u2705 Saved relaxed structure material ID: {relaxed_material_saved.id}")
_relaxed_saved_response

## References

[1] mat3ra-made: https://github.com/Exabyte-io/made  
[2] ASE calculators overview: https://wiki.fysik.dtu.dk/ase/ase/calculators/calculators.html  
[3] MACE-MP-0 foundation models: https://github.com/ACEsuit/mace?tab=readme-ov-file#foundation-models